# 00 - Setup: Live Or Offline

This notebook checks the AcmeCloud fixture and initializes the shared Metatate client.

Offline mode is the default. It reads committed response fixtures and requires no Snowflake account.

Live mode calls the Snowflake-managed Metatate MCP server. Set `METATATE_EXAMPLES_MODE=live`, configure `.env`, and export the PAT before starting Jupyter.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

client = get_client()
mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
print(f"Metatate examples mode: {mode}")


def decision_label(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("decision") or decision.get("overall_decision") or data.get("overall_decision", "UNKNOWN")
    return decision or data.get("overall_decision", "UNKNOWN")


def rationale_text(response):
    data = response.get("data", {})
    decision = data.get("decision")
    if isinstance(decision, dict):
        return decision.get("rationale") or data.get("summary") or ""
    return data.get("rationale") or data.get("summary") or ""


def agent_action_text(response):
    action = response.get("data", {}).get("agent_action", {})
    if isinstance(action, dict):
        return action.get("message") or action.get("safe_next_step") or action.get("suggested_next_tool") or ""
    return ""


## Load Synthetic Tables


In [ ]:
table_dir = repo_root / "sample-data" / "acmecloud" / "tables"
tables = {}
for path in sorted(table_dir.glob("*.csv")):
    tables[path.stem] = pd.read_csv(path)
    print(f"{path.name}: {len(tables[path.stem])} rows")


In [ ]:
tables["customers"].head()


## Discover Governed Context


In [ ]:
response = client.discover_context(database="ACMECLOUD_DEMO", schema="PUBLIC")
print(json.dumps(response["data"], indent=2))
